# T11: LLM-Agenten

`ToolAgent` (`conversational_toolkit/agents/tool_agent.py`) automatisiert den Ablauf aus T10: das LLM aufrufen, die angeforderten Tools ausführen, ihre Ergebnisse anhängen und das wiederholen, bis das LLM ohne Tool-Call antwortet oder `max_steps` LLM-Aufrufe aufgebraucht sind. Wirft ein Tool eine Exception, geht die Fehlermeldung an das LLM zurück, statt den Agenten anzuhalten.

Dieses Notebook baut einen Agenten mit zwei Tools und dann denselben Agenten mit strukturierter JSON-Ausgabe.

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Agent mit zwei Tools

Der Agent entscheidet pro Frage, ob er kein Tool, eines oder mehrere nacheinander verwendet. Die Zeilen `Tool ... returned` zeigen, welche Tools gelaufen sind.

`ToolAgent.answer` nimmt eine `QueryWithContext` (Frage plus Verlauf) statt einer Liste von Nachrichten. `ask` ruft `answer` auf und gibt die Antwort aus.

System-Prompt übersetzt:

> Du bist ein hilfreicher Assistent. Antworte kurz.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM

from ki_kompass_rag.baseline_rag import ask
from ki_kompass_rag.tool_agents import HalfBoldText, SumTwoNumbers

logger.remove()
logger.add(sys.stderr, level="WARNING")
# Print one line per tool call
logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.agents.tool_agent",
    format="{message}",
)

llm = OpenAILLM(tools=[SumTwoNumbers(), HalfBoldText()], tool_choice="auto")
agent = ToolAgent(
    system_prompt="You are a helpful assistant. Answer briefly.", llm=llm, max_steps=5
)

queries = [
    "How are you?",
    "Sum 2 and 59.",
    "Bold every other word: 'This is an example of the half bold text tool in action.'",
    "Sum 10 and 20, then write one sentence about the result with every other word bolded.",
]
for query in queries:
    print(f"\n>>> {query}")
    await ask(agent, query)

## Strukturierte Ausgabe

`response_format` mit einem strikten JSON Schema zwingt die endgültige Antwort in ein Objekt mit `answer` und `explanations`, sodass sich das Ergebnis zuverlässig parsen lässt. Tool-Calls funktionieren wie zuvor; nur die letzte Nachricht muss dem Schema entsprechen.

System-Prompt übersetzt:

> Antworte wie ein Pirat. Begründe jede Verwendung eines Tools. Halte dich kurz.

In [ ]:
answer_schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string", "description": "The answer in markdown."},
        "explanations": {
            "type": "array",
            "description": "Justifications for the answer.",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 3,
        },
    },
    "required": ["answer", "explanations"],
    "additionalProperties": False,
}

structured_llm = OpenAILLM(
    tools=[SumTwoNumbers(), HalfBoldText()],
    tool_choice="auto",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "AnswerSchema",
            "schema": answer_schema,
            "strict": True,
        },
    },
)
structured_agent = ToolAgent(
    system_prompt="Answer like a pirate. Justify every tool use. Keep it short.",
    llm=structured_llm,
    max_steps=5,
)

for query in queries:
    print(f"\n>>> {query}")
    answer = await structured_agent.answer(QueryWithContext(query=query, history=[]))
    data = json.loads(answer.content[0].text or "")
    print(json.dumps(data, indent=2, ensure_ascii=False))

## Konversation

Frühere Runden kommen in `history`. Gespeichert werden nur die Fragen und die endgültigen Antworten, nicht die Tool-Calls dazwischen.

In [ ]:
history = [
    LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=queries[-1])]
    ),
    answer,
]
follow_up = await structured_agent.answer(
    QueryWithContext(query="Make a joke about our conversation.", history=history)
)
data = json.loads(follow_up.content[0].text or "")
print(json.dumps(data, indent=2, ensure_ascii=False))